This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [ ]:
import great_expectations as gx
import logging

In [ ]:
from great_expectations_experimental.expectations.expect_queried_custom_query_to_return_num_rows import ExpectQueriedCustomQueryToReturnNumRows

In [ ]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [ ]:
context = gx.get_context(context_root_dir=gx_context_root_dir)
context.list_expectation_suites()

In [ ]:
import yaml

In [ ]:
from datetime import date,datetime

In [ ]:
logging.basicConfig(level=logging.DEBUG, force = True)

In [ ]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [ ]:
datasource_config.get("project")

In [ ]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [ ]:
gx_datasource.get_asset_names()

In [ ]:
context.list_expectation_suite_names()

In [ ]:
TABLE_NAME = "port_visits"
DUMMY_BATCH_DATE = '2023-04-01' # the date slice you want to run interactive expectations on

In [ ]:
for current_expectation_suite_name in [es for es in context.list_expectation_suite_names() if TABLE_NAME in es and 'constraints' in es]:
    current_expectation_suite_name = [es for es in context.list_expectation_suite_names() if TABLE_NAME in es and 'constraints' in es][0]
    print(current_expectation_suite_name)
    current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
    current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        br_options={DATE_PARTITION_COLUMN: DUMMY_BATCH_DATE}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)
    gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)
    gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)

    gx_validator.expect_column_values_to_be_unique('visit_id')
    gx_validator.expect_column_values_to_not_be_null('visit_id')

    gx_validator.expect_column_values_to_be_in_set('confidence', [1, 2, 3, 4])

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT COUNT(*), vessel_id, start_timestamp
        FROM {{active_batch}}
        GROUP BY vessel_id, start_timestamp
        HAVING COUNT(*) > 1
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "There should only be one port_visit for a vessel at the same time.",
            }
    })
    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT COUNT(*), vessel_id, end_timestamp
        FROM {{active_batch}}
        GROUP BY vessel_id, end_timestamp
        HAVING COUNT(*) > 1
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "There should only be one port_visit for a vessel at the same time.",
            }
    })
    
    gx_validator.save_expectation_suite(discard_failed_expectations=False)